In [1]:
import cv2
import os
import numpy as np

INPUT_DIR = "dataset"
OUTPUT_DIR = "processed_dataset"
IMG_SIZE = 512

os.makedirs(OUTPUT_DIR, exist_ok=True)

def preprocess_image(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    img = cv2.GaussianBlur(img, (5, 5), 0)
    img = cv2.medianBlur(img, 5)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0

    return img

for label in os.listdir(INPUT_DIR):
    in_dir = os.path.join(INPUT_DIR, label)
    out_dir = os.path.join(OUTPUT_DIR, label)
    os.makedirs(out_dir, exist_ok=True)

    for f in os.listdir(in_dir):
        img_path = os.path.join(in_dir, f)
        processed = preprocess_image(img_path)
        cv2.imwrite(os.path.join(out_dir, f),
                    (processed * 255).astype(np.uint8))

print("✅ Preprocessing completed")


✅ Preprocessing completed


In [2]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
import cv2
import numpy as np

base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(512, 512, 3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)

model = Model(inputs=base_model.input, outputs=x)

def extract_densenet_features(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (512, 512))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = preprocess_input(img)
    img = np.expand_dims(img, axis=0)

    features = model.predict(img, verbose=0)
    return features.flatten()   # 1024 features


In [3]:
from skimage.feature import local_binary_pattern

def extract_lbp_features(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (512, 512))

    lbp = local_binary_pattern(img, P=8, R=1, method="uniform")
    hist, _ = np.histogram(lbp.ravel(),
                           bins=np.arange(0, 59),
                           density=True)
    return hist


In [4]:
from skimage.feature import graycomatrix, graycoprops

def extract_glcm_features(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (512, 512))

    glcm = graycomatrix(
        img,
        distances=[1],
        angles=[0],
        levels=256,
        symmetric=True,
        normed=True
    )

    return np.array([
        graycoprops(glcm, 'contrast')[0, 0],
        graycoprops(glcm, 'correlation')[0, 0],
        graycoprops(glcm, 'energy')[0, 0],
        graycoprops(glcm, 'homogeneity')[0, 0]
    ])


In [5]:
def extract_hybrid_features(img_path):
    f1 = extract_densenet_features(img_path)
    f2 = extract_lbp_features(img_path)
    f3 = extract_glcm_features(img_path)
    return np.concatenate([f1, f2, f3])


Label name	Label ID	Cancer status
Normal	0	❌ Non-cancer
Bengin (Benign)	1	⚠️ Non-cancer (tumor but not cancer)
Malignant	2	✅ Cancer

In [6]:
DATASET_DIR = "processed_dataset"
LABELS = {"Normal": 0, "Bengin": 1, "Malignant": 2}

X = []
y = []

for label_name, label_id in LABELS.items():
    folder = os.path.join(DATASET_DIR, label_name)

    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)
        features = extract_hybrid_features(img_path)

        X.append(features)
        y.append(label_id)

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1097, 1086)
y shape: (1097,)


In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [8]:
import random
from deap import base, creator, tools, algorithms
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

NUM_FEATURES = X.shape[1]

# Avoid duplicate creation in Jupyter
if "FitnessMax" not in creator.__dict__:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat,
                 creator.Individual, toolbox.attr_bool, NUM_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate(individual):
    idx = [i for i, bit in enumerate(individual) if bit == 1]
    if len(idx) == 0:
        return 0,

    X_sel = X[:, idx]
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    acc = cross_val_score(clf, X_sel, y, cv=5).mean()

    penalty = len(idx) / NUM_FEATURES
    return acc - 0.1 * penalty,

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.02)
toolbox.register("select", tools.selTournament, tournsize=3)

population = toolbox.population(n=30)

algorithms.eaSimple(
    population,
    toolbox,
    cxpb=0.5,
    mutpb=0.2,
    ngen=25,
    verbose=True
)

best = tools.selBest(population, k=1)[0]
selected_features = [i for i, bit in enumerate(best) if bit == 1]

X_optimized = X[:, selected_features]

print("✅ Selected Features:", len(selected_features))


gen	nevals
0  	30    
1  	13    
2  	23    
3  	17    
4  	16    
5  	19    
6  	13    
7  	19    
8  	18    
9  	23    
10 	14    
11 	20    
12 	19    
13 	15    
14 	17    
15 	10    
16 	21    
17 	18    
18 	18    
19 	18    
20 	18    
21 	16    
22 	18    
23 	17    
24 	9     
25 	17    
✅ Selected Features: 508


In [9]:
y_binary = np.where(y == 2, 1, 0)

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_optimized,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)


In [11]:
import joblib

# Save scaler
joblib.dump(scaler, "scaler.pkl")

# Save selected features from GA
joblib.dump(selected_features, "selected_features.pkl")

print("✅ PKL files saved")

✅ PKL files saved


In [12]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X_optimized,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

input_dim = X_train.shape[1]

# Build DNN model
dnn = tf.keras.Sequential([
    
    tf.keras.layers.Dense(512, activation="relu", input_shape=(input_dim,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

dnn.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

dnn.summary()

history = dnn.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=40,
    batch_size=32
)

C:\Users\kikik\AppData\Roaming\Python\Python310\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │       260,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 428,033 (1.63 MB)

 Trainable params: 426,497 (1.63 MB)

 Non-trainable params: 1,536 (6.00 KB)

Epoch 1/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.9236 - loss: 0.1908 - val_accuracy: 0.9909 - val_loss: 0.0198
Epoch 2/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9829 - loss: 0.0434 - val_accuracy: 0.9909 - val_loss: 0.0141
Epoch 3/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9989 - loss: 0.0091 - val_accuracy: 0.9955 - val_loss: 0.0141
Epoch 4/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0033 - val_accuracy: 0.9955 - val_loss: 0.0153
Epoch 5/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9954 - loss: 0.0115 - val_accuracy: 0.9909 - val_loss: 0.0203
Epoch 6/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.0031 - val_accuracy: 1.0000 - val_loss: 0.0051
Epoch 7/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.0026 - val_accuracy: 1.0000 - val_loss: 0.0035
Epoch 8/40
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0029 - val_accuracy: 0.9955 - v

In [13]:
loss, acc = dnn.evaluate(X_test, y_test)

print("DNN Accuracy:", acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9955 - loss: 0.0241     
DNN Accuracy: 0.9954545497894287


In [14]:
dnn.save("lung_cancer_dnn.h5")

print("✅ DNN model saved")

✅ DNN model saved


In [15]:
import tensorflow as tf

model = tf.keras.models.load_model("lung_cancer_dnn.h5")

print("Model loaded successfully")

Model loaded successfully


In [ ]:



import cv2
import numpy as np
import tensorflow as tf
import joblib

from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D

from skimage.feature import local_binary_pattern
from skimage.feature import graycomatrix, graycoprops


# ===============================
# Load trained model
# ===============================

model = tf.keras.models.load_model("lung_cancer_dnn.h5")
scaler = joblib.load("scaler.pkl")
selected_features = joblib.load("selected_features.pkl")
print("✅ Model loaded successfully")

# ===============================
# DenseNet Feature Extractor
# ===============================

base_model = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(512, 512, 3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)

feature_model = Model(inputs=base_model.input, outputs=x)


def extract_densenet_features(img_path):

    img = cv2.imread(img_path)
    img = cv2.resize(img, (512, 512))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = preprocess_input(img)
    img = np.expand_dims(img, axis=0)

    features = feature_model.predict(img, verbose=0)

    return features.flatten()


# ===============================
# LBP Features
# ===============================

def extract_lbp_features(img_path):

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (512, 512))

    lbp = local_binary_pattern(img, P=8, R=1, method="uniform")

    hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, 59),
        density=True
    )

    return hist


# ===============================
# GLCM Features
# ===============================

def extract_glcm_features(img_path):

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (512, 512))

    glcm = graycomatrix(
        img,
        distances=[1],
        angles=[0],
        levels=256,
        symmetric=True,
        normed=True
    )

    return np.array([
        graycoprops(glcm, 'contrast')[0, 0],
        graycoprops(glcm, 'correlation')[0, 0],
        graycoprops(glcm, 'energy')[0, 0],
        graycoprops(glcm, 'homogeneity')[0, 0]
    ])


# ===============================
# Hybrid Feature Vector
# ===============================

def extract_hybrid_features(img_path):

    f1 = extract_densenet_features(img_path)
    f2 = extract_lbp_features(img_path)
    f3 = extract_glcm_features(img_path)

    return np.concatenate([f1, f2, f3])


# ===============================
# Prediction Function
# ===============================

def predict_image(img_path):

    features = extract_hybrid_features(img_path)

    features = scaler.transform([features])

    features = features[:, selected_features]

    pred = model.predict(features)[0][0]

    print("Raw Model Output:", pred)

    if pred > 0.5:
        predicted_class = "Malignant"
        confidence = pred
    else:
        predicted_class = "Non-Cancer (Normal / Benign)"
        confidence = 1 - pred

    print("Prediction:", predicted_class)
    print("Confidence:", round(confidence * 100, 2), "%")

    if predicted_class == "Malignant":
        print("⚠️ Cancer Detected")
    else:
        print("✅ Non-Cancer")


# ===============================
# Test Prediction
# ===============================

img_path = r"D:\Local Disk (D)\Lung Project 100%\lung\first\dataset\Malignant\Malignant case (1).jpg"

predict_image(img_path)

# img_path = r"D:\Local Disk (D)\Lung Project 100%\lung\first\datasetA\Normal\Normal case (1).jpg"

# predict_image(img_path)

✅ Model loaded successfully


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step
Raw Model Output: 1.0
Prediction: Malignant
Confidence: 100.0 %
⚠️ Cancer Detected
